# Exploratory Data Analysis (EDA) & Dataset Audit
## Binary Person Segmentation on the Penn-Fudan Benchmark

**Course / Lab:** Machine Learning Project, IMPACT Lab, Dept. of CSE, IIT Patna  
**Objective:** Scientifically audit, explore, and visualize the Penn-Fudan Pedestrian dataset prior to model training, verifying zero-leakage partitions and class distributions in compliance with Step 3–5 of the Project Guidelines.

In [1]:
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Libraries loaded successfully.")

### 1. Load Dataset Audit Summary and Split Manifest
We load the audit summary generated by `person_segmentation.analyze_dataset` and the frozen 70/15/15 partition manifest (`seed 42`).

In [2]:
summary_path = Path("../report/dataset_analysis/summary.json")
split_path = Path("../report/dataset_analysis/split_seed42.json")

with open(summary_path, "r") as f:
    summary = json.load(f)

with open(split_path, "r") as f:
    splits = json.load(f)

print(f"Total Images: {summary['dataset']['total_images']}")
print(f"Total Pedestrian Instances: {summary['dataset']['total_instances']}")
print(f"Train Split: {len(splits['train'])} scenes ({len(splits['train'])/170*100:.1f}%)")
print(f"Validation Split: {len(splits['val'])} scenes ({len(splits['val'])/170*100:.1f}%)")
print(f"Held-Out Test Split: {len(splits['test'])} scenes ({len(splits['test'])/170*100:.1f}%)")

### 2. Verify Zero Data Leakage Across Splits
We verify that the partition sets are pairwise mutually disjoint with zero filename overlap.

In [3]:
set_train = set(splits['train'])
set_val = set(splits['val'])
set_test = set(splits['test'])

assert len(set_train.intersection(set_val)) == 0, "Leakage between Train and Val!"
assert len(set_train.intersection(set_test)) == 0, "Leakage between Train and Test!"
assert len(set_val.intersection(set_test)) == 0, "Leakage between Val and Test!"
print("PASS: Verified zero overlap across Train, Validation, and Test partitions.")

### 3. Image Dimensions and Resolution Variance
Penn-Fudan photographs are captured under natural outdoor conditions with varying aspect ratios and resolutions.

In [4]:
per_image_csv = Path("../report/dataset_analysis/per_image.csv")
df = pd.read_csv(per_image_csv)
df[['width', 'height', 'instances', 'person_pixel_ratio']].describe().round(4)

### 4. Class Balance & Foreground Prevalence
In binary person segmentation, the foreground class exhibits substantial spatial imbalance. Background dominates roughly 82.72% of total pixels, while human silhouettes occupy only 17.28% on average.

In [5]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))

# Histogram of Person Pixel Ratios
ax[0].hist(df['person_pixel_ratio'] * 100, bins=20, color='#2563EB', edgecolor='black', alpha=0.8)
ax[0].set_title("Distribution of Foreground Person Pixels (%)", fontsize=11, fontweight='bold')
ax[0].set_xlabel("Foreground % per Image")
ax[0].set_ylabel("Number of Images")

# Bar chart of Pedestrian Instances per Image
inst_counts = df['instances'].value_counts().sort_index()
ax[1].bar(inst_counts.index, inst_counts.values, color='#0D9488', edgecolor='black', alpha=0.8)
ax[1].set_title("Pedestrian Instances per Photographic Scene", fontsize=11, fontweight='bold')
ax[1].set_xlabel("Number of Pedestrians")
ax[1].set_ylabel("Number of Images")

plt.tight_layout()
plt.show()

### 5. Key EDA Takeaways for Model & Loss Design
1. **Class Imbalance:** Because foreground pixels average only 17.28%, a naive model predicting all zeros achieves ~82.7% accuracy while delivering 0.0 IoU. Consequently, we employ a joint loss combining Binary Cross-Entropy with Soft Dice ($L_{total} = 0.5 L_{BCE} + 0.5 L_{Dice}$).
2. **Resizing Hygiene:** Images vary from $348 \times 425$ to $628 \times 1017$. Bilinear interpolation is used for RGB photographs, while nearest-neighbor interpolation is strictly required for masks to prevent fractional label bleeding.
3. **Small Sample Overfitting:** With only 119 training scenes, controlled spatial inductive biases (horizontal flip and bounded zoom-in) are crucial to prevent memorization and stabilize multi-seed training variance.